# 02 · Sensibilidad del Decision Transformer al retorno objetivo
**Autor:** Daniel Sebastian Velasco Munar · Aprendizaje Profundo 202630 · PUJ

El Decision Transformer recibe como entrada el retorno que se le pide obtener. En el notebook 01 se le pidió siempre el
retorno medio del conjunto expert. Aquí se cargan los modelos ya entrenados (carpeta `modelos` en Drive, semillas 1 y 2)
y se evalúan pidiendo distintas fracciones de ese retorno, sin volver a entrenar.

> **Antes de ejecutar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`. Tarda unos 25 minutos.

In [ ]:
!pip install -q "gymnasium[mujoco]" mujoco
!pip install -q "minari[all]"
!pip install -q transformers

In [ ]:
import os, sys, json, glob, time, warnings
warnings.filterwarnings("ignore")
os.environ["MUJOCO_GL"] = "egl"
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, minari
from transformers import DecisionTransformerConfig, DecisionTransformerModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/proyecto_decision_transformer"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REFS = json.load(open(f"{OUT}/referencias.json"))
K, RTG_SCALE, MAX_EP_LEN = 20, 1000.0, 1000
FRACCIONES = [0.25, 0.5, 0.75, 1.0, 1.25]   # fracción del retorno medio del conjunto expert
SEMILLAS = [1, 2]
EPISODIOS = 5
CSV = f"{OUT}/retorno_objetivo.csv"
normalized = lambda env_name, R: 100 * (R - REFS[env_name]["random"]) / (REFS[env_name]["expert"] - REFS[env_name]["random"])
print(DEVICE, "· modelos encontrados:", len(glob.glob(f"{OUT}/modelos/*.pt")))

In [ ]:
class DTAgent:
    # Igual que en el notebook 01: acciones autorregresivas con los últimos K pasos
    def __init__(self, model, mean, std, sd, ad, target):
        self.model, self.mean, self.std, self.sd, self.ad, self.target = model, mean, std, sd, ad, target
    def reset(self):
        self.S, self.A, self.R = [], [], []; self.G = [self.target / RTG_SCALE]; self.t = 0
    @torch.no_grad()
    def act(self, obs):
        sd, ad = self.sd, self.ad
        self.S.append(((obs - self.mean) / self.std).astype(np.float32)); self.A.append(np.zeros(ad, np.float32)); self.R.append(0.0)
        L = min(K, len(self.S)); pad = K - L
        def seq(x, dim):
            x = np.asarray(x[-L:], np.float32).reshape(L, dim)
            return np.concatenate([np.zeros((pad, dim), np.float32), x])
        ts = np.concatenate([np.zeros(pad, np.int64), np.minimum(np.arange(self.t - L + 1, self.t + 1), MAX_EP_LEN - 1)])
        m = np.concatenate([np.zeros(pad), np.ones(L)]).astype(np.float32)
        to = lambda x: torch.tensor(x[None], device=DEVICE)
        out = self.model(states=to(seq(self.S, sd)), actions=to(seq(self.A, ad)), rewards=to(seq(self.R, 1)),
                         returns_to_go=to(seq(self.G, 1)), timesteps=to(ts), attention_mask=to(m), return_dict=True)
        a = out.action_preds[0, -1].cpu().numpy(); self.A[-1] = a
        return np.clip(a, -1, 1)
    def update(self, r):
        self.R[-1] = r; self.G.append(self.G[-1] - r / RTG_SCALE); self.t += 1

def evaluate(env, agent, n=EPISODIOS, seed0=10_000):
    rets = []
    for k in range(n):
        obs, _ = env.reset(seed=seed0 + k); agent.reset(); done, R = False, 0.0
        while not done:
            obs, r, term, trunc, _ = env.step(agent.act(obs)); agent.update(r); R += r; done = term or trunc
        rets.append(R)
    return np.array(rets)

In [ ]:
done = set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV); done = set(zip(prev["Robot"], prev["Datos"], prev["Semilla"], prev["Fracción"]))
t0 = time.time()
for env_name in ["hopper", "halfcheetah"]:
    env = minari.load_dataset(f"mujoco/{env_name}/expert-v0", download=True).recover_environment()
    for q in ["simple", "medium", "expert"]:
        for seed in SEMILLAS:
            f = f"{OUT}/modelos/dt_{env_name}_{q}_{seed}.pt"
            if not os.path.exists(f):
                print("falta", f); continue
            ck = torch.load(f, map_location=DEVICE, weights_only=False)
            cfg = DecisionTransformerConfig.from_dict(ck["config"])
            model = DecisionTransformerModel(cfg).to(DEVICE); model.load_state_dict(ck["pesos"]); model.eval()
            for fr in FRACCIONES:
                if (env_name, q, seed, fr) in done: continue
                target = fr * REFS[env_name]["expert"]
                rets = evaluate(env, DTAgent(model, ck["media"], ck["desv"], cfg.state_dim, cfg.act_dim, target))
                row = {"Robot": env_name, "Datos": q, "Semilla": seed, "Fracción": fr, "Retorno pedido": target,
                       "Retorno medio": rets.mean(), "Puntaje normalizado": normalized(env_name, rets.mean())}
                pd.DataFrame([row]).to_csv(CSV, mode="a", header=not os.path.exists(CSV), index=False)
                print(f"{env_name:<11} {q:<6} semilla {seed} · {fr:4.2f}× → puntaje {row['Puntaje normalizado']:6.1f}"
                      f" · {(time.time() - t0)/60:4.1f} min")
    env.close()
print("Listo")

In [ ]:
res = pd.read_csv(CSV)
tabla = res.groupby(["Robot", "Datos", "Fracción"])["Puntaje normalizado"].mean().unstack("Fracción").round(1)
display(tabla)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, env_name in zip(axes, ["hopper", "halfcheetah"]):
    for q in ["simple", "medium", "expert"]:
        s = res[(res.Robot == env_name) & (res.Datos == q)].groupby("Fracción")["Puntaje normalizado"].mean()
        ax.plot(s.index * 100, s.values, "o-", label=q)
    ax.plot([25, 125], [25, 125], "k:", lw=0.8, label="puntaje = pedido")
    ax.set_title(env_name); ax.set_xlabel("Retorno pedido (% del expert)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Puntaje normalizado"); axes[0].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/retorno_objetivo.png", dpi=150); plt.show()

In [ ]:
# Libera la GPU al terminar para no gastar unidades
from google.colab import runtime
runtime.unassign()